In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

import random
self_seed = 0
g = torch.Generator().manual_seed(self_seed)

import math

In [2]:
block_sizes = [3,4,6]       # 上下文长度
embedding_dims = [2,5,10]   # 嵌入维度
hidden_dim = 100    # 隐藏层大小
batch_size = 32     # 批次数
lr_base = 0.1       # 初始学习率

In [3]:
pinyins = open('data/names1/names1_pinyin_components.txt','r').read().splitlines()
pinyins_to_chars = open('data/names1/pinyin_components_to_hanzi_chars1.txt','r').read().splitlines()

tokens = [p.split() for p in pinyins]
token_list = sorted({'.'} | {t for token in tokens for t in token})
token_list_len = len(token_list)
stoi = {s:i for i,s in enumerate(token_list)}
itos = {i:s for i,s in enumerate(token_list)}

In [4]:
def build_dataset(data,block_size):
    X,Y = [],[]
    for t in data:
        # print(['.']*block_size + t + ['.'])
        # print(t + ['.'])

        str1 = ['.']*block_size + t + ['.']
        str2 = t + ['.']

        i=0
        for _,y in zip(str1,str2):
            # print(str1[i:i+block_size],y)
            # print([stoi[str1[j]] for j in range(i,i+block_size)],stoi[y])
            X.append([stoi[str1[j]] for j in range(i,i+block_size)])
            Y.append(stoi[y])
            i+=1


    X=torch.tensor(X)
    Y=torch.tensor(Y)
    #print(X.shape,Y.shape,X.dtype,Y.dtype)
    return X,Y

In [5]:
def get_lr(lr_base, i, max_steps, warmup_steps=10000, lr_min=1e-5):
    """
    余弦退火学习率
    warmup_steps: 预热步数，前1万步线性增加
    lr_min: 最低学习率，防止后期完全不动
    """
    # 1. 预热阶段：从 0 线性增加到 lr_base
    if i < warmup_steps:
        return lr_base * (i / warmup_steps)
    
    # 2. 余弦退火阶段：从 lr_base 平滑衰减到 lr_min
    # 注意要减去预热步数
    progress = (i - warmup_steps) / (max_steps - warmup_steps)
    # 使用 cosine 公式
    return lr_min + 0.5 * (lr_base - lr_min) * (1.0 + math.cos(math.pi * progress))

In [6]:
def init_model(embedding_dim,block_size):
    E = torch.randn((token_list_len, embedding_dim)        ,generator=g)
    W1 = torch.randn((block_size*embedding_dim, hidden_dim),generator=g) * (5/3)/((embedding_dim*block_size)**0.5)*0.4
    b1 = torch.randn(hidden_dim                            ,generator=g) * 0.2
    W2 = torch.randn((hidden_dim,token_list_len)           ,generator=g) * 0.01
    b2 = torch.randn(token_list_len                        ,generator=g) * 0
    return E,W1,b1,W2,b2

@torch.no_grad()
def evaluate(X, Y, E, W1, b1, W2, b2, block_size, embedding_dim):
    emb = E[X]
    h = torch.tanh(emb.view(-1, block_size * embedding_dim) @ W1 + b1)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Y)
    pred = logits.argmax(dim=1)
    acc = (pred == Y).float().mean()

    return loss.item(), acc.item()

In [ ]:
random.seed(self_seed)
tokens_shuffled = tokens[:]
random.shuffle(tokens_shuffled)
n1 = int(0.8*len(tokens_shuffled))
n2 = int(0.9*len(tokens_shuffled))
train_tokens = tokens_shuffled[:n1]
test_tokens = tokens_shuffled[n2:]

all_lossi = []
results = []

for block_size in block_sizes:
    for embedding_dim in embedding_dims:
        # print(block_size,embedding_dim)
        # 训练集 80% -> 优化模型权重
        # 测试集 10% -> 测试模型泛化性和准确率
        Xtr,Ytr = build_dataset(train_tokens,block_size)
        Xte,Yte = build_dataset(test_tokens,block_size)

        E,W1,b1,W2,b2 = init_model(embedding_dim,block_size)

        parameters=[E,W1,b1,W2,b2]
        for p in parameters:
            p.requires_grad = True

        # train
        max_steps = 200000
        lossi = []
        for i in range(max_steps):
            # forward pass
            batch_idxs = torch.randint(0,Xtr.shape[0],(batch_size,))
            emb = E[Xtr[batch_idxs]]
            hpreact = emb.view(-1,block_size*embedding_dim)@W1+b1 # h_practivate
            h = torch.tanh(hpreact)
            logits = h@W2+b2
            loss = F.cross_entropy(logits,Ytr[batch_idxs])

            #backward pass
            for p in parameters:
                p.grad = None
            loss.backward()

            # update
            lr = get_lr(lr_base,i,max_steps)
            for p in parameters:
                p.data += -lr*p.grad
            
            # track stats
            lossi.append(loss.item())

            # print
            if(i%50000==0):
                print(f'[{block_size,embedding_dim}]-> {i:6d}/{max_steps:6d}: loss={loss.item()}')
            #break

        all_lossi.append({
            'block_size': block_size,
            'embedding_dim': embedding_dim,
            'lossi': lossi,
        })

        train_loss, train_acc = evaluate(Xtr, Ytr, E, W1, b1, W2, b2, block_size, embedding_dim)
        test_loss, test_acc = evaluate(Xte, Yte, E, W1, b1, W2, b2, block_size, embedding_dim)
        results.append({
            'block_size': block_size,
            'embedding_dim': embedding_dim,
            'train_loss': train_loss,
            'test_loss': test_loss,
            'train_acc': train_acc,
            'test_acc': test_acc,
            'gap': test_loss - train_loss,
        })
        print(f'[{block_size,embedding_dim}] train_loss={train_loss:.4f}, test_loss={test_loss:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')
    


[(3, 2)]->      0/200000: loss=4.065091609954834
[(3, 2)]->  50000/200000: loss=1.2935421466827393
[(3, 2)]-> 100000/200000: loss=1.606410026550293
[(3, 2)]-> 150000/200000: loss=1.6117677688598633
[(3, 2)] train_loss=1.3418, test_loss=1.3681, train_acc=0.5288, test_acc=0.5189
[(3, 5)]->      0/200000: loss=4.059160232543945
[(3, 5)]->  50000/200000: loss=1.614160180091858
[(3, 5)]-> 100000/200000: loss=1.2928072214126587
[(3, 5)]-> 150000/200000: loss=1.0454434156417847
[(3, 5)] train_loss=1.3321, test_loss=1.3678, train_acc=0.5297, test_acc=0.5192
[(3, 10)]->      0/200000: loss=4.060723304748535
[(3, 10)]->  50000/200000: loss=1.3646836280822754
[(3, 10)]-> 100000/200000: loss=1.7651034593582153
[(3, 10)]-> 150000/200000: loss=1.2575857639312744
[(3, 10)] train_loss=1.3268, test_loss=1.3734, train_acc=0.5302, test_acc=0.5166
[(4, 2)]->      0/200000: loss=4.061738014221191
[(4, 2)]->  50000/200000: loss=1.3408056497573853
[(4, 2)]-> 100000/200000: loss=1.448192834854126
[(4, 2)]-> 1

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.sort_values('test_loss')

In [ ]:
plot_df = df.sort_values(['block_size', 'embedding_dim'])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# loss: 训练集和测试集越接近，说明泛化差距越小
for block_size, group in plot_df.groupby('block_size'):
    group = group.sort_values('embedding_dim')
    axes[0].plot(group['embedding_dim'], group['train_loss'], marker='o', linestyle='--', label=f'train bs={block_size}')
    axes[0].plot(group['embedding_dim'], group['test_loss'], marker='o', label=f'test bs={block_size}')

axes[0].set_title('Loss vs Embedding Dim')
axes[0].set_xlabel('embedding_dim')
axes[0].set_ylabel('loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

# accuracy: 给定上下文时，下一个 token 预测正确的比例
for block_size, group in plot_df.groupby('block_size'):
    group = group.sort_values('embedding_dim')
    axes[1].plot(group['embedding_dim'], group['train_acc'], marker='o', linestyle='--', label=f'train bs={block_size}')
    axes[1].plot(group['embedding_dim'], group['test_acc'], marker='o', label=f'test bs={block_size}')

axes[1].set_title('Accuracy vs Embedding Dim')
axes[1].set_xlabel('embedding_dim')
axes[1].set_ylabel('accuracy')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

# gap: test_loss - train_loss，越大越可能过拟合
for block_size, group in plot_df.groupby('block_size'):
    group = group.sort_values('embedding_dim')
    axes[2].plot(group['embedding_dim'], group['gap'], marker='o', label=f'bs={block_size}')

axes[2].axhline(0, color='black', linewidth=1, alpha=0.5)
axes[2].set_title('Generalization Gap')
axes[2].set_xlabel('embedding_dim')
axes[2].set_ylabel('test_loss - train_loss')
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in [
    (axes[0], 'test_loss', 'Test Loss Heatmap'),
    (axes[1], 'test_acc', 'Test Accuracy Heatmap'),
]:
    pivot = df.pivot(index='block_size', columns='embedding_dim', values=metric)
    im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
    ax.set_title(title)
    ax.set_xlabel('embedding_dim')
    ax.set_ylabel('block_size')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f'{pivot.values[i, j]:.3f}', ha='center', va='center', color='white')

    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()